In [ ]:
import requests
import os
import zipfile
import pandas as pd
import glob
import duckdb



In [ ]:
os.makedirs('dados_bolsa_familia', exist_ok=True)

# Simular um navegador para evitar bloqueio
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "pt-BR,pt;q=0.9",
    "Referer": "https://www.portaldatransparencia.gov.br/",
}

downloads = {
    2017: "https://www.portaltransparencia.gov.br/download-de-dados/bolsa-familia-pagamentos/201701",
    2018: "https://www.portaltransparencia.gov.br/download-de-dados/bolsa-familia-pagamentos/201801",
    2019: "https://www.portaltransparencia.gov.br/download-de-dados/bolsa-familia-pagamentos/201901",
    2020: "https://www.portaltransparencia.gov.br/download-de-dados/bolsa-familia-pagamentos/202001",
    2021: "https://www.portaltransparencia.gov.br/download-de-dados/bolsa-familia-pagamentos/202101",
    2023: "https://www.portaldatransparencia.gov.br/download-de-dados/novo-bolsa-familia/202303",
    2024: "https://www.portaldatransparencia.gov.br/download-de-dados/novo-bolsa-familia/202401",
    2025: "https://www.portaldatransparencia.gov.br/download-de-dados/novo-bolsa-familia/202501",
    2026: "https://www.portaldatransparencia.gov.br/download-de-dados/novo-bolsa-familia/202601",
}

for ano, url in downloads.items():
    print(f"Baixando {ano}...", end=" ")
    try:
        response = requests.get(url, headers=headers, timeout=120)
        if response.status_code == 200:
            zip_path = f"dados_bolsa_familia/{ano}.zip"
            with open(zip_path, 'wb') as f:
                f.write(response.content)
            with zipfile.ZipFile(zip_path, 'r') as z:
                z.extractall('dados_bolsa_familia/')
            os.remove(zip_path)
            print("OK")
        else:
            print(f"Erro HTTP {response.status_code}")
    except Exception as e:
        print(f"Falhou: {e}")

print("\nPronto! Arquivos em ./dados_bolsa_familia/")

In [ ]:

pasta = 'dados_bolsa_familia'

# Listar todos os CSVs encontrados
arquivos = glob.glob(f'{pasta}/*.csv')
print(f"Arquivos encontrados: {len(arquivos)}")
for a in arquivos:
    print(" -", os.path.basename(a))

In [ ]:
for arquivo in sorted(arquivos):
    df_temp = pd.read_csv(arquivo, sep=';', encoding='latin-1', nrows=2)
    print(f"\n📄 {os.path.basename(arquivo)}")
    print(f"   Colunas: {list(df_temp.columns)}")

In [ ]:

con = duckdb.connect('dados_bolsa_familia/unificado.duckdb')

con.execute("""
    CREATE TABLE bolsa_familia AS
    SELECT * FROM read_csv_auto(
        'dados_bolsa_familia/*.csv',
        delim=';',
        header=true,
        encoding='latin-1'
    )
""")

print(con.execute("SELECT COUNT(*) FROM bolsa_familia").fetchone())
con.close()